This is a simulation based on the Fraunhofer diffraction theory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.signal import find_peaks

plt.rcParams['text.usetex'] = True

wavelength = 632e-9
W = 2e-3
H = 4e-3
z = 0.5

x_range = np.linspace(-2e-3, 2e-3, 500)
y_range = np.linspace(-2e-3, 2e-3, 500)
x, y = np.meshgrid(x_range, y_range)

beta_x = (np.pi * W * x) / (wavelength * z)
beta_y = (np.pi * H * y) / (wavelength * z)

intensity = (np.sinc(beta_x / np.pi)**2) * (np.sinc(beta_y / np.pi)**2)

scaling_factor = 1000
intensity_rescaled = intensity * scaling_factor

profile_x = np.sum(intensity_rescaled, axis=0)
profile_y = np.sum(intensity_rescaled, axis=1)

x_range_mm = x_range * 1e3
y_range_mm = y_range * 1e3

threshold = 0.05
peaks_x, _ = find_peaks(profile_x, height=threshold)
peaks_y, _ = find_peaks(profile_y, height=threshold)

if len(peaks_x) > 1:
    avg_x_order_distance = np.mean(np.diff(x_range_mm[peaks_x[:5]]))

if len(peaks_y) > 1:
    avg_y_order_distance = np.mean(np.diff(y_range_mm[peaks_y[:5]]))

fig, ax = plt.subplots(2, 2, figsize=(14, 10))

padding = 0.5
ax[0, 0].imshow(np.ones_like(x), cmap='binary', extent=[-W/2 * 1e3 - padding, W/2 * 1e3 + padding, -H/2 * 1e3 - padding, H/2 * 1e3 + padding])
ax[0, 0].set_title(f'Rectangular Aperture', fontsize=12)
ax[0, 0].set_xlabel('$W$ ($mm$)', fontsize=10)
ax[0, 0].set_ylabel('$H$ ($mm$)', fontsize=10)
ax[0, 0].add_patch(Rectangle((-W/2 * 1e3, -H/2 * 1e3), W * 1e3, H * 1e3, linewidth=2, edgecolor='black', facecolor='none'))
ax[0, 0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)

ax[0, 1].imshow(np.log10(intensity_rescaled + 1), cmap='gist_heat', extent=[x_range_mm[0], x_range_mm[-1], y_range_mm[0], y_range_mm[-1]])
ax[0, 1].set_title(f'Diffraction Pattern', fontsize=12)
ax[0, 1].set_xlabel('$x$ ($mm$)', fontsize=10)
ax[0, 1].set_ylabel('$y$ ($mm$)', fontsize=10)

ax[1, 0].plot(x_range_mm, profile_x, color='red')
ax[1, 0].set_title(f'$x$-profile', fontsize=12)
ax[1, 0].set_xlabel('$x$ ($mm$)', fontsize=10)
ax[1, 0].set_ylabel('Intensity', fontsize=10)
ax[1, 0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)

ax[1, 1].plot(y_range_mm, profile_y, color='green')
ax[1, 1].set_title(f'$y$-profile', fontsize=12)
ax[1, 1].set_xlabel('$y$ ($mm$)', fontsize=10)
ax[1, 1].set_ylabel('Intensity', fontsize=10)
ax[1, 1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)

simulation_text = f'''
Simulation Parameters:
$W = {W*1e3:.1f} \\ \\mathrm{{mm}}$
$H = {H*1e3:.1f} \\ \\mathrm{{mm}}$
$\\lambda = {wavelength*1e9:.1f} \\ \\mathrm{{nm}}$
$z = {z:.1f} \\ \\mathrm{{m}}$

Average Order Distance:
$x= {avg_x_order_distance:.2f} \\ mm$
$y= {avg_y_order_distance:.2f} \\ mm$

Diffraction Function:
$I(x,y) \\propto \\left(\\beta_x  \\right)^2 \\times \\left( \\beta_y \\right)^2$

$\\beta_x= \\frac{{\\sin{{\\left( \\frac{{\\pi Wx}}{{\\lambda z}} \\right)}}}}{{\\frac{{\\pi Wx}}{{\\lambda z}}}}$
$\\beta_y= \\frac{{\\sin{{\\left( \\frac{{\\pi Hy}}{{\\lambda z}} \\right)}}}}{{\\frac{{\\pi Hy}}{{\\lambda z}}}}$
'''

fig.text(0.5, 0.60, simulation_text, ha='center', fontsize=12, fontweight='light',
         bbox=dict(facecolor='white', edgecolor='lightgrey', boxstyle='round, pad=1'))

plt.subplots_adjust(top=0.92, bottom=0.12, left=0.1, right=0.9)
plt.tight_layout()
plt.show()
